# Chronos-2 autoregressive benchmark
Run in a Colab GPU runtime. This notebook uses the repository's VAR simulator and forecasts **yield levels jointly across ten maturities**. Run the small smoke experiment first, then the full sweep.

The oracle knows the true generator coefficients but receives only the observed context. It is an ideal reference, not a fitted competitor. The null (`persistence=0`) has no predictable direction. Positive MSE skill means an improvement over repeating the last observed curve. Finite-sample oracle skill can be negative.

Results are scored at each **exact lead**, not averaged over all preceding minutes. The 80% interval coverage and three-quantile pinball loss assess marginal uncertainty, not joint curve calibration. Current Chronos-2 source returns the median in its output named `mean`; we explicitly select the 0.5 quantile.

API reference: https://github.com/amazon-science/chronos-forecasting/blob/main/src/chronos/chronos2/pipeline.py


In [ ]:
%pip install -q "chronos-forecasting[extras]>=2.2"


In [ ]:
# The remote repository must contain autoregressive_benchmark.py.
# Alternatively upload that file and synthetic_yield_curve_var.py to /content
# and skip this cell (run imports from their upload directory).
!git clone https://github.com/stewaala/chronos-timeseries.git /content/chronos-timeseries
%cd /content/chronos-timeseries


In [ ]:
import importlib.metadata
import json
import platform
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from chronos import BaseChronosPipeline
from autoregressive_benchmark import parameter_grid, run_benchmark, summarize

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)
print("chronos-forecasting:", importlib.metadata.version("chronos-forecasting"))
if "pipeline" not in globals():
    pipeline = BaseChronosPipeline.from_pretrained("amazon/chronos-2", device_map=device)


In [ ]:
# Smoke run: two configurations, two forecast origins each.
smoke = run_benchmark(
    pipeline,
    configs=[dict(name="null", n=1, persistence=0., lag_decay=0.7),
             dict(name="persistent", n=5, persistence=0.8, lag_decay=0.7)],
    seeds=[11], context_length=512, horizon=10,
    origins_per_seed=2, curves_per_batch=2,
)
display(summarize(smoke).query("lead in [1, 5, 10]"))


## Main experiment
The grid contains 16 configurations: one null plus persistence 0.2/0.5/0.8, orders 1/5/20, and lag decay 0.3/0.9 (only one decay for order 1).

Defaults yield 480 forecast origins, with 10 maturities and 60 forecast steps each. GPU batches contain four independent curves; reduce `curves_per_batch` if GPU memory is limited. `cross_learning=False` prevents information sharing between rolling origins. All ten maturities within a curve are still forecast jointly.

`one_minute` holds unconditional minute-change covariance fixed as persistence changes. `long_run` instead holds long-run cumulative covariance fixed. Keep these experiments separate. Match comparisons using skill scores, since difficulty and long-horizon volatility change with persistence.

Tenors and rolling origins are dependent. Seeds are the independent replications; use more seeds to confirm apparent improvements, and fresh seeds after choosing parameters. This is an exploratory sweep, not a significance test. A single forecast to 60 minutes is sliced by lead; separate calls with other requested horizons may behave differently.


In [ ]:
CONFIGS = parameter_grid()
SETTINGS = dict(
    seeds=list(range(10)),
    context_length=4096,
    horizon=60,
    origins_per_seed=3,
    curves_per_batch=4,
    volatility_match="one_minute",
    end="2026-01-01 00:00:00+00:00",
)
print("Configurations:", len(CONFIGS))
print("Forecast origins:", len(CONFIGS) * len(SETTINGS["seeds"]) * SETTINGS["origins_per_seed"])
results = run_benchmark(pipeline, CONFIGS, **SETTINGS)
summary = summarize(results)
by_seed = summarize(results, by=("config", "seed", "lead"))
by_tenor = summarize(results, by=("config", "tenor", "lead"))

display(summary.query("lead in [1, 5, 15, 60]")[[
    "config", "lead", "model", "rmse_bp", "mae_bp",
    "mse_skill_vs_naive", "coverage_80", "width_80_bp", "pinball_bp"
]])


In [ ]:
# MSE skill by exact lead. Oracle separation from zero measures available signal.
fig, axes = plt.subplots(4, 4, figsize=(17, 13), sharex=True)
for ax, config in zip(axes.flat, CONFIGS):
    subset = summary[summary.config == config["name"]]
    for model in ["chronos", "oracle"]:
        points = subset[subset.model == model]
        ax.plot(points.lead, points.mse_skill_vs_naive, label=model)
    ax.axhline(0, color="black", linewidth=0.8)
    ax.set_title(config["name"])
    ax.set_xlabel("Forecast lead (minutes)")
    ax.set_ylabel("MSE skill vs naive")
    ax.grid(alpha=0.2)
axes.flat[0].legend()
fig.tight_layout()
plt.show()

# Spread across independent seeds: descriptive variation, not confidence intervals.
seed_spread = by_seed.query("model == 'chronos' and lead in [1, 5, 15, 60]")
display(seed_spread.groupby(["config", "lead"]).mse_skill_vs_naive.agg(
    ["mean", "std", "min", "max"]
))


In [ ]:
# Save to the Colab runtime. Download these files or copy them to Drive to retain them.
output = Path("var_benchmark_results")
output.mkdir(exist_ok=True)
results.to_csv(output / "raw_metrics.csv.gz", index=False)
summary.to_csv(output / "summary.csv", index=False)
by_seed.to_csv(output / "by_seed.csv", index=False)
by_tenor.to_csv(output / "by_tenor.csv", index=False)
metadata = dict(configs=CONFIGS, settings=SETTINGS,
                model="amazon/chronos-2", device=device,
                model_commit=getattr(pipeline.model.config, "_commit_hash", None),
                python=platform.python_version(),
                packages={name: importlib.metadata.version(name) for name in
                          ["chronos-forecasting", "torch", "numpy", "pandas", "scipy"]})
(output / "settings.json").write_text(json.dumps(metadata, indent=2))
print("Saved to", output.resolve())
